# Estudo de Caso 4.X1 — Solver Colebrook-White: Newton-Raphson e Diagrama de Moody

| Campo | Detalhe |
|:------|:--------|
| **Capítulo Associado** | Capítulo 4 — Escoamento em Tubulações e Bombeamento |
| **Nível** | Pós-Graduação |
| **Tempo estimado** | 45 minutos |
| **Autor** | Jader Lugon Junior |
| **Método numérico** | Newton-Raphson para equação não-linear implícita |
| **Bibliografia** | Lugon Jr. et al. (2026), Cap. 4; White (2016), Cap. 6; Colebrook (1939); Swamee & Jain (1976) |

---

## 📋 Resumo

Implementamos um solver iterativo baseado no método de Newton-Raphson para resolver a equação implícita de Colebrook-White, que determina o fator de atrito $f$ em escoamentos turbulentos. A solução é validada contra a correlação explícita de Swamee-Jain e a equação de Haaland. O Diagrama de Moody é gerado interativamente, e a convergência quadrática do método é demonstrada. Uma análise de sensibilidade investiga o efeito da rugosidade relativa $\varepsilon/D$ e do número de Reynolds $Re$ sobre o fator de atrito.

---

## 🎯 Objetivos de Aprendizagem

Ao final deste estudo de caso, o leitor será capaz de:

1. **Formular** a equação de Colebrook-White como problema de raiz não-linear
2. **Implementar** o método de Newton-Raphson com transformação de variável
3. **Comparar** a solução iterativa com correlações explícitas (Swamee-Jain, Haaland)
4. **Analisar** a convergência quadrática do método de Newton-Raphson
5. **Construir** o Diagrama de Moody para diferentes rugosidades relativas
6. **Avaliar** a sensibilidade do fator de atrito a $Re$ e $\varepsilon/D$

---
## ⚙️ Seção 0: Configuração Inicial

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL — Importação de bibliotecas
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown
import time

# Configuração de estilo dos gráficos (padrão do guia)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print('✅ Ambiente configurado com sucesso')
print(f'📦 NumPy {np.__version__} | Matplotlib {plt.matplotlib.__version__}')

---
## ⚙️ Seção 1: Parâmetros do Problema

### Contexto Físico

Em escoamento turbulento em tubulações, o fator de atrito $f$ determina a perda de carga por unidade de comprimento:

$$\Delta p = f \cdot \frac{L}{D} \cdot \frac{\rho U^2}{2}$$

A equação de Colebrook-White (1939) correlaciona $f$ com $Re$ e a rugosidade relativa $\varepsilon/D$, porém é **implícita** em $f$: o fator de atrito aparece em ambos os lados da equação.

> 💡 **Observação:** A equação de Colebrook-White é baseada em dados experimentais de Nikuradse (rugosidade uniforme) e Colebrook (rugosidade comercial). É válida para $4000 < Re < 10^8$ e $0 < \varepsilon/D < 0{,}05$.

### Tabela de Parâmetros

| Parâmetro | Símbolo | Valor | Unidade |
|:----------|:-------:|------:|:-------:|
| Número de Reynolds (caso base) | $Re$ | 10⁵ | – |
| Rugosidade relativa (caso base) | $\varepsilon/D$ | 10⁻⁴ | – |
| Tolerância de convergência | $\epsilon_{tol}$ | 10⁻⁸ | – |
| Máximo de iterações | $N_{iter,max}$ | 100 | – |
| Faixa de Re (Moody) | $Re$ | 3×10³ – 10⁸ | – |
| Faixa de $\varepsilon/D$ (Moody) | $\varepsilon/D$ | 10⁻⁶ – 10⁻¹ | – |

In [ ]:
# ============================================================
# PARÂMETROS DO PROBLEMA
# ============================================================

# Caso base
Re_base = 1.0e5       # Número de Reynolds [-]
eps_D_base = 1.0e-4   # Rugosidade relativa [-]

# Parâmetros do solver
tol = 1e-8            # Tolerância de convergência [-]
max_iter = 100        # Máximo de iterações [-]

# Faixas para o Diagrama de Moody
Re_range = np.logspace(3.5, 8, 200)         # 3000 a 10^8
eps_D_values = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 5e-2]  # Rugosidades

print('📏 Parâmetros do Problema:')
print(f'  • Caso base: Re = {Re_base:.0e}, ε/D = {eps_D_base:.1e}')
print(f'  • Tolerância: {tol:.1e}')
print(f'  • Máximo de iterações: {max_iter}')
print(f'  • Faixa de Re (Moody): {Re_range[0]:.0e} a {Re_range[-1]:.0e}')
print(f'  • Rugosidades relativas: {len(eps_D_values)} valores')

---
## 🔬 Seção 2: Verificação — Correlações Explícitas de Referência

Antes da implementação do solver iterativo, estabelecemos correlações explícitas como referência de validação.

### Swamee & Jain (1976)

$$f = \frac{0{,}25}{\left[\log_{10}\left(\frac{\varepsilon/D}{3{,}7} + \frac{5{,}74}{Re^{0,9}}\right)\right]^2}$$

### Haaland (1983)

$$\frac{1}{\sqrt{f}} = -1{,}8 \log_{10}\left[\left(\frac{\varepsilon/D}{3{,}7}\right)^{1,11} + \frac{6{,}9}{Re}\right]$$

> 📖 **Referência:** Swamee & Jain (1976); Haaland (1983); White (2016), Cap. 6, Seção 6.6.

In [ ]:
# ============================================================
# FUNÇÕES DE REFERÊNCIA — Correlações Explícitas
# ============================================================

def fator_atrito_swamee_jain(Re, eps_D):
    """
    Fator de atrito pela correlação explícita de Swamee-Jain (1976).

    Parameters
    ----------
    Re : float ou ndarray
        Número de Reynolds [-].
    eps_D : float
        Rugosidade relativa [-].

    Returns
    -------
    float ou ndarray
        Fator de atrito de Darcy [-].

    Notes
    -----
    Válido para 5000 < Re < 10^8 e 10^-6 < eps/D < 10^-2.
    Erro típico: ±1% em relação a Colebrook-White.
    """
    return 0.25 / (np.log10(eps_D / 3.7 + 5.74 / Re**0.9))**2


def fator_atrito_haaland(Re, eps_D):
    """
    Fator de atrito pela correlação explícita de Haaland (1983).

    Parameters
    ----------
    Re : float ou ndarray
        Número de Reynolds [-].
    eps_D : float
        Rugosidade relativa [-].

    Returns
    -------
    float ou ndarray
        Fator de atrito de Darcy [-].

    Notes
    -----
    Válido para 4000 < Re < 10^8.
    Erro típico: ±2% em relação a Colebrook-White.
    """
    inv_sqrt_f = -1.8 * np.log10((eps_D / 3.7)**1.11 + 6.9 / Re)
    return 1.0 / inv_sqrt_f**2


print('✅ Correlações explícitas de referência implementadas')

# Verificação rápida
f_SJ = fator_atrito_swamee_jain(Re_base, eps_D_base)
f_HL = fator_atrito_haaland(Re_base, eps_D_base)

print(f'\n📊 Verificação rápida (Re={Re_base:.0e}, ε/D={eps_D_base:.1e}):')
print(f'  • Swamee-Jain: f = {f_SJ:.6f}')
print(f'  • Haaland:     f = {f_HL:.6f}')
print(f'  • Diferença:   {abs(f_SJ - f_HL)/f_SJ*100:.3f}%')

---
## 📐 Seção 3: Formulação Matemática

### 3.1 Equação de Colebrook-White (1939)

$$\frac{1}{\sqrt{f}} = -2 \log_{10}\left(\frac{\varepsilon/D}{3{,}7} + \frac{2{,}51}{Re \sqrt{f}}\right)$$

onde:
- $f$ = fator de atrito de Darcy [-]
- $Re$ = número de Reynolds [-]
- $\varepsilon/D$ = rugosidade relativa [-]

### 3.2 Transformação de Variável

Definindo $x = 1/\sqrt{f}$, a equação se torna:

$$F(x) = x + 2 \log_{10}\left(\frac{\varepsilon/D}{3{,}7} + \frac{2{,}51}{Re \cdot x}\right) = 0$$

### 3.3 Método de Newton-Raphson

$$x^{(k+1)} = x^{(k)} - \frac{F(x^{(k)})}{F'(x^{(k)})}$$

A derivada analítica é:

$$F'(x) = 1 - \frac{2}{\ln(10)} \cdot \frac{2{,}51}{Re \cdot x^2} \cdot \frac{1}{\frac{\varepsilon/D}{3{,}7} + \frac{2{,}51}{Re \cdot x}}$$

### 3.4 Critério de Convergência

$$|x^{(k+1)} - x^{(k)}| < \epsilon_{tol}$$

> 📖 **Referência:** Colebrook (1939); Lugon Jr. et al. (2026), Cap. 4, Seção 4.3.

---
## ⚡ Seção 4: Critério Numérico e Convergência

> 💡 **Observação:** Neste problema de equação algébrica não-linear, não se aplica a condição CFL. Os critérios de adequação são:

1. **Convergência quadrática:** O método de Newton-Raphson converge quadraticamente quando iniciado próximo da raiz. Isso significa que o número de dígitos corretos aproximadamente dobra a cada iteração.

2. **Estimativa inicial:** Usamos Swamee-Jain como chute inicial, que está tipicamente a < 2% da solução exata.

3. **Domínio de validade:** A equação de Colebrook-White é válida para escoamento turbulento ($Re > 4000$). Para regime laminar, $f = 64/Re$.

4. **Tolerância:** $\epsilon_{tol} = 10^{-8}$ garante precisão de pelo menos 6 dígitos significativos em $f$.

---
## 🌪️ Seção 5: Condições de Entrada (Forçamento)

Neste problema algébrico, o forçamento são os **parâmetros de entrada**:

- Número de Reynolds $Re$ (determinado por $U$, $D$, $\nu$)
- Rugosidade relativa $\varepsilon/D$ (propriedade do material do tubo)

### Valores típicos de rugosidade absoluta

| Material | $\varepsilon$ [mm] |
|:---------|:---:|
| Vidro, plástico | 0 |
| Aço comercial | 0,045 |
| Ferro fundido | 0,26 |
| Concreto | 0,3 – 3,0 |
| Ferro fundido corroído | 1,0 – 1,5 |

> 💡 **Observação:** Para uma tubulação de aço de D = 100 mm, $\varepsilon/D = 0{,}045/100 = 4{,}5 \times 10^{-4}$.

---
## 🧮 Seção 6: Implementação Numérica

In [ ]:
# ============================================================
# SOLVER: Colebrook-White por Newton-Raphson
# ============================================================

def colebrook_newton(Re, eps_D, tol=1e-8, max_iter=100):
    """
    Resolve a equação de Colebrook-White por Newton-Raphson.

    Parameters
    ----------
    Re : float
        Número de Reynolds [-].
    eps_D : float
        Rugosidade relativa [-].
    tol : float
        Tolerância de convergência [-].
    max_iter : int
        Máximo de iterações [-].

    Returns
    -------
    tuple
        (f, historico, n_iter) onde f é o fator de atrito,
        historico é a lista de valores de f por iteração,
        e n_iter é o número de iterações.

    Notes
    -----
    Transformação de variável: x = 1/sqrt(f)
    F(x) = x + 2*log10(eps_D/3.7 + 2.51/(Re*x)) = 0
    """
    # Estimativa inicial (Swamee-Jain)
    f = fator_atrito_swamee_jain(Re, eps_D)
    x = 1.0 / np.sqrt(f)

    historico = [f]

    for i in range(max_iter):
        # F(x)
        F = x + 2.0 * np.log10(eps_D / 3.7 + 2.51 / (Re * x))

        # F'(x) — derivada analítica
        argumento = eps_D / 3.7 + 2.51 / (Re * x)
        F_linha = 1.0 - (2.0 / np.log(10.0)) * (2.51 / (Re * x**2)) / argumento

        # Newton-Raphson
        x_novo = x - F / F_linha

        # Critério de convergência
        if abs(x_novo - x) < tol:
            f_novo = 1.0 / x_novo**2
            historico.append(f_novo)
            return f_novo, historico, i + 1

        x = x_novo
        f = 1.0 / x**2
        historico.append(f)

    raise RuntimeError(f'Colebrook não convergiu em {max_iter} iterações')


print('✅ Solver de Colebrook-White (Newton-Raphson) implementado')

In [ ]:
# ============================================================
# CASO BASE — Solução e Validação
# ============================================================

t_inicio = time.time()

f_newton, hist, n_iter = colebrook_newton(Re_base, eps_D_base, tol, max_iter)

t_cpu_solver = time.time() - t_inicio

# Referências
f_SJ = fator_atrito_swamee_jain(Re_base, eps_D_base)
f_HL = fator_atrito_haaland(Re_base, eps_D_base)

# Erros relativos
erro_SJ = abs(f_newton - f_SJ) / f_newton * 100
erro_HL = abs(f_newton - f_HL) / f_newton * 100

print('=' * 60)
print('  CASO BASE: SOLUÇÃO E VALIDAÇÃO')
print('=' * 60)
print(f'\n📊 Parâmetros: Re = {Re_base:.0e}, ε/D = {eps_D_base:.1e}')
print(f'\n🔢 Resultados:')
print(f'  • Colebrook-White (Newton): f = {f_newton:.8f}')
print(f'  • Swamee-Jain:              f = {f_SJ:.8f} (erro = {erro_SJ:.4f}%)')
print(f'  • Haaland:                  f = {f_HL:.8f} (erro = {erro_HL:.4f}%)')
print(f'\n⚙️ Desempenho:')
print(f'  • Número de iterações: {n_iter}')
print(f'  • Tempo de CPU: {t_cpu_solver*1e6:.1f} μs')
print(f'\n📈 Histórico de convergência:')
for i, f_val in enumerate(hist):
    print(f'  • Iteração {i}: f = {f_val:.10f}')

> 💡 **Observação:**
> - O método de Newton-Raphson converge em apenas 3-4 iterações (convergência quadrática).
> - Swamee-Jain e Haaland têm erro < 1% em relação à solução exata.
> - Para projetos preliminares, Swamee-Jain é suficiente; para projetos detalhados, Colebrook-White.

---
## 📊 Seção 7: Visualização

In [ ]:
# ============================================================
# GRÁFICO 1: Convergência do Newton-Raphson
# ============================================================

fig, ax = plt.subplots(figsize=(10, 5))

iteracoes = np.arange(len(hist))
ax.semilogy(iteracoes, hist, 'bo-', linewidth=2, markersize=8,
            label='Histórico de Newton-Raphson')
ax.axhline(f_newton, color='r', linestyle='--', linewidth=1.5,
           label=f'f convergido = {f_newton:.6f}')

ax.set_xlabel('Iteração', fontsize=12)
ax.set_ylabel('Fator de atrito, f', fontsize=12)
ax.set_title('Convergência do Método de Newton-Raphson',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

> 💡 **Observação:**
> - A convergência é **quadrática**: o erro cai de ~10⁻³ para ~10⁻⁶ em uma iteração, e para ~10⁻¹² na seguinte.
> - O chute inicial (Swamee-Jain) já está a < 0,1% da solução final.
> - Em escala logarítmica, a queda é visível como uma descida abrupta.

In [ ]:
# ============================================================
# GRÁFICO 2: DIAGRAMA DE MOODY
# ============================================================

fig, ax = plt.subplots(figsize=(14, 9))

# Regime laminar: f = 64/Re
Re_laminar = np.linspace(2000, 4000, 50)
ax.loglog(Re_laminar, 64.0 / Re_laminar, 'k-', linewidth=2.5,
          label='Laminar: f = 64/Re')

# Regime turbulento: Colebrook-White para diferentes ε/D
cores = plt.cm.viridis(np.linspace(0, 1, len(eps_D_values)))

for eps_D, cor in zip(eps_D_values, cores):
    f_values = []
    for Re_val in Re_range:
        if Re_val < 4000:
            f_values.append(64.0 / Re_val)
        else:
            f_val, _, _ = colebrook_newton(Re_val, eps_D)
            f_values.append(f_val)
    ax.loglog(Re_range, f_values, color=cor, linewidth=2,
              label=f'ε/D = {eps_D:.0e}')

# Linha de tubo liso (ε/D → 0)
f_liso = []
for Re_val in Re_range:
    if Re_val < 4000:
        f_liso.append(64.0 / Re_val)
    else:
        f_val, _, _ = colebrook_newton(Re_val, 1e-8)
        f_liso.append(f_val)
ax.loglog(Re_range, f_liso, 'k--', linewidth=1.5, label='Tubo liso')

ax.set_xlabel('Número de Reynolds, Re', fontsize=13)
ax.set_ylabel('Fator de atrito de Darcy, f', fontsize=13)
ax.set_title('Diagrama de Moody', fontsize=15, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3, which='both')
ax.set_xlim(2000, 1e8)
ax.set_ylim(0.008, 0.1)

# Marcar caso base
ax.plot(Re_base, f_newton, 'r*', markersize=15, zorder=5,
        label=f'Caso base: Re={Re_base:.0e}, f={f_newton:.4f}')

plt.tight_layout()
plt.show()

> 💡 **Observação:**
> - No regime laminar (Re < 2300), $f = 64/Re$ independe da rugosidade.
> - No regime turbulento, $f$ depende tanto de $Re$ quanto de $\varepsilon/D$.
> - Para $Re$ muito elevado, $f$ tende a um valor constante que depende apenas de $\varepsilon/D$ (regime completamente rugoso).
> - A zona de transição (2300 < Re < 4000) é evitada em projetos de engenharia.

---
## 📈 Seção 8: Análise Quantitativa

In [ ]:
# ============================================================
# TABELA COMPARATIVA DOS MÉTODOS
# ============================================================

dados_tabela = {
    'Método': [
        'Colebrook-White (Newton-Raphson)',
        'Swamee-Jain (explícito)',
        'Haaland (explícito)'
    ],
    'f': [
        f'{f_newton:.8f}',
        f'{f_SJ:.8f}',
        f'{f_HL:.8f}'
    ],
    'Erro vs. Newton [%]': [
        '0,0000 (referência)',
        f'{erro_SJ:.4f}',
        f'{erro_HL:.4f}'
    ],
    'Iterações': [
        f'{n_iter}',
        '0 (explícito)',
        '0 (explícito)'
    ],
    'Indicação de uso': [
        'Projeto detalhado, alta precisão',
        'Projeto preliminar, rapidez',
        'Projeto preliminar, rapidez'
    ]
}

df_comp = pd.DataFrame(dados_tabela)
display(Markdown('### Comparação dos Métodos'))
display(Markdown(df_comp.to_markdown(index=False)))

In [ ]:
# ============================================================
# MÉTRICAS DE DESEMPENHO DO SOLVER
# ============================================================

# Testar em múltiplos pontos
Re_test = np.array([5e3, 1e4, 5e4, 1e5, 5e5, 1e6, 1e7])
eps_D_test = np.array([1e-6, 1e-5, 1e-4, 1e-3, 1e-2])

n_iter_total = 0
n_testes = 0
tempo_total = 0.0

for Re_val in Re_test:
    for eps_val in eps_D_test:
        t_ini = time.time()
        f_val, _, n_it = colebrook_newton(Re_val, eps_val)
        tempo_total += time.time() - t_ini
        n_iter_total += n_it
        n_testes += 1

print('📈 MÉTRICAS DE DESEMPENHO DO SOLVER')
print('=' * 50)
print(f'  • Número de testes: {n_testes}')
print(f'  • Média de iterações: {n_iter_total/n_testes:.1f}')
print(f'  • Tempo médio por chamada: {tempo_total/n_testes*1e6:.1f} μs')
print(f'  • Tempo total: {tempo_total*1e3:.2f} ms')
print(f'\n✅ Solver robusto em toda a faixa de Re e ε/D testada')

---
## 🔍 Seção 9: Análise de Sensibilidade

Analisamos a sensibilidade do fator de atrito a dois parâmetros:
1. **Rugosidade relativa $\varepsilon/D$:** efeito do material e envelhecimento
2. **Número de Reynolds $Re$:** efeito da velocidade e diâmetro

In [ ]:
# ============================================================
# SENSIBILIDADE — f vs Re para diferentes ε/D
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Painel 1: f vs Re para diferentes ε/D
Re_sens = np.logspace(4, 7, 100)
for eps_D, cor in zip(eps_D_values, plt.cm.viridis(np.linspace(0, 1, len(eps_D_values)))):
    f_values = [colebrook_newton(Re_val, eps_D)[0] for Re_val in Re_sens]
    axes[0].loglog(Re_sens, f_values, color=cor, linewidth=2, label=f'ε/D = {eps_D:.0e}')

axes[0].set_xlabel('Número de Reynolds, Re', fontsize=12)
axes[0].set_ylabel('Fator de atrito, f', fontsize=12)
axes[0].set_title('(a) Efeito da Rugosidade Relativa', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, which='both')

# Painel 2: f vs ε/D para diferentes Re
eps_D_sens = np.logspace(-6, -1, 100)
Re_fixed = [1e4, 1e5, 1e6, 1e7]
for Re_val, cor in zip(Re_fixed, plt.cm.plasma(np.linspace(0, 1, len(Re_fixed)))):
    f_values = [colebrook_newton(Re_val, eps_val)[0] for eps_val in eps_D_sens]
    axes[1].loglog(eps_D_sens, f_values, color=cor, linewidth=2, label=f'Re = {Re_val:.0e}')

axes[1].set_xlabel('Rugosidade relativa, ε/D', fontsize=12)
axes[1].set_ylabel('Fator de atrito, f', fontsize=12)
axes[1].set_title('(b) Efeito do Número de Reynolds', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

> 💡 **Observação:**
> - **(a)** Para $Re$ elevado, $f$ tende a um patamar que depende apenas de $\varepsilon/D$ (regime completamente rugoso).
> - **(b)** Para $Re$ baixo, a rugosidade tem pouco efeito; para $Re$ elevado, o efeito é significativo.
> - A sensibilidade de $f$ a $\varepsilon/D$ aumenta com $Re$: em altas velocidades, a escolha do material é crítica.
> - Para tubulações envelhecidas (corrosão), $\varepsilon/D$ pode aumentar 10-100×, elevando significativamente as perdas de carga.

---
## 💡 Seção 10: Discussão

### Interpretação Física dos Resultados

1. **Convergência quadrática:** O método de Newton-Raphson converge em 3-4 iterações porque a transformação $x = 1/\sqrt{f}$ torna a função quase linear. Isso é muito mais eficiente que métodos de busca (bissecção) que exigiriam ~30 iterações para a mesma precisão.

2. **Correlações explícitas:** Swamee-Jain e Haaland são aproximações polinomiais da equação de Colebrook-White com erro < 1-2%. Para projetos preliminares e simulações em tempo real, são preferíveis pela velocidade. Para projetos detalhados e calibração, Colebrook-White é a referência.

3. **Regime completamente rugoso:** Para $Re$ muito elevado, o termo $2{,}51/(Re\sqrt{f})$ torna-se desprezível, e $f$ depende apenas de $\varepsilon/D$. Isso explica por que tubulações antigas têm perdas de carga muito maiores que as novas.

4. **Implicações de projeto:** Um erro de 10% em $f$ pode resultar em erro de 10% na perda de carga, o que pode levar a subdimensionamento de bombas ou tubulações inadequadas.

### Limitações do Modelo

- **Escoamento turbulento apenas:** A equação de Colebrook-White não é válida para $Re < 4000$. Para regime laminar, $f = 64/Re$.
- **Rugosidade uniforme:** A equação assume rugosidade distribuída uniformemente. Rugosidade localizada (juntas, soldas) não é capturada.
- **Sem dependência do tempo:** Não considera envelhecimento, corrosão ou deposição ao longo do tempo.
- **Tubo circular:** A equação é válida apenas para seções circulares. Para outras seções, usar diâmetro hidráulico.
- **Fluido newtoniano:** Não é aplicável a fluidos não-newtonianos.

### Aplicações Práticas

- **Engenharia hidráulica:** Dimensionamento de adutoras, redes de distribuição e sistemas de bombeamento.
- **Engenharia química:** Projeto de trocadores de calor, reatores e tubulações de processo.
- **Engenharia sanitária:** Dimensionamento de redes de esgoto e drenagem.
- **Simulação em tempo real:** Modelos de redes hidráulicas (EPANET) usam Swamee-Jain pela velocidade.

### Conexão com a Teoria

> 📖 **Referência:** A equação de Colebrook-White é descrita em Colebrook (1939) e White (2016), Cap. 6, Seção 6.6. O método de Newton-Raphson para equações não-lineares segue Lugon Jr. et al. (2026), Cap. 4, Seção 4.3. As correlações de Swamee-Jain e Haaland são apresentadas em White (2016), Cap. 6.

---
## 🔄 Seção 11: Navegação

- [📚 Voltar ao Capítulo 4](../notebooks/04_Escoamento_Tubulacoes.ipynb)
- [📂 Outros Estudos de Caso](README.md)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">
Estudo de Caso 4.X1

Parte da coleção *"Fenômenos de Transporte: Fundamentos e Modelagem Computacional"*
</div>